# 08 — Debugging broken links

Companion to **[Chapter 14](../14-debugging-broken-links.md)**.

The source table has **8 rows**. Your graph has **6 operation nodes**.
The transformation reported success. Find all four discrepancies.

## Setup

In [ ]:
# ---------------------------------------------------------------- setup ----
import os
from pathlib import Path

from cognite.client import CogniteClient, global_config
global_config.disable_pypi_version_check = True
from cognite.client.config import ClientConfig
from cognite.client.credentials import OAuthClientCredentials, OAuthInteractive

# Find the repo root by its markers, so this cell works wherever Jupyter started.
HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents]
             if (p / "pyproject.toml").exists() and (p / "training").exists()), HERE)

env_path = ROOT / ".env"
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s or s.startswith("#") or "=" not in s:
            continue
        k, v = s.split("=", 1)
        if " #" in v and not v.startswith(('"', "'")):
            v = v.split(" #", 1)[0].rstrip()
        os.environ.setdefault(k, v)      # a real environment variable always wins

missing = [k for k in ("CDF_PROJECT", "CDF_CLUSTER", "IDP_CLIENT_ID")
           if not os.environ.get(k)]
assert not missing, f"Missing {missing}. Copy .env.example to {env_path} and fill it in."


def cdf_client(name: str) -> CogniteClient:
    """Build the client EXPLICITLY.

    `CogniteClient()` with no arguments does not read your .env. The SDK removed
    implicit construction in v8 and raises:
        ValueError: No ClientConfig has been provided
    The branch below is the two-identity rule from Chapter 02, in code.
    """
    base_url = os.environ.get("CDF_URL") or f"https://{os.environ['CDF_CLUSTER']}.cognitedata.com"
    scopes = [s for s in os.environ.get("IDP_SCOPES", f"{base_url}/.default").split(",") if s]

    if os.environ.get("LOGIN_FLOW", "interactive").lower() == "interactive":
        creds = OAuthInteractive(              # you, in a browser -- needs
            authority_url=os.environ["IDP_AUTHORITY_URL"],   # localhost:53000
            client_id=os.environ["IDP_CLIENT_ID"],           # as a redirect URI
            scopes=scopes)
    else:
        creds = OAuthClientCredentials(        # unattended: a service principal
            token_url=os.environ["IDP_TOKEN_URL"],
            client_id=os.environ["IDP_CLIENT_ID"],
            client_secret=os.environ["IDP_CLIENT_SECRET"],
            scopes=scopes)

    return CogniteClient(ClientConfig(
        client_name=name, project=os.environ["CDF_PROJECT"],
        base_url=base_url, credentials=creds))


YOURNAME = os.environ.get("PARTICIPANT", "YOURNAME")   # [CHANGE] if not in .env
client   = cdf_client(f"dm-handson-{YOURNAME}-debug")

space       = f"isp_{YOURNAME}_TRN"
schema_edm  = f"ssp_{YOURNAME}_TrainingCore_edm"
schema_sdm  = f"ssp_{YOURNAME}_MaintenanceInsight_sdm"
raw_db      = f"rwd_{YOURNAME}_Training_TRN"
model_version = "v1.0.0"


# --- identifiers every chapter uses ---------------------------------------
from cognite.client.data_classes.data_modeling import ViewId
from cognite.client.data_classes import filters as flt
from cognite.client.data_classes.data_modeling.query import (
    Query, QuerySync, NodeResultSetExpression, EdgeResultSetExpression,
    Select, SourceSelector)
from cognite.client.data_classes.data_modeling import (
    NodeId, EdgeId, NodeApply, EdgeApply, NodeOrEdgeData, DirectRelationReference)
from cognite.client.data_classes.raw import RowWrite

from cognite.client.data_classes.aggregations import Count, Avg, Max

INSTANCE_SPACE = space
EDM_SPACE      = schema_edm
SDM_SPACE      = schema_sdm
RAW_DB         = raw_db
MODEL_VERSION  = model_version

ASSET      = ViewId("cdf_cdm", "CogniteAsset",     "v1")
EQUIPMENT  = ViewId("cdf_cdm", "CogniteEquipment", "v1")
ACTIVITY   = ViewId("cdf_cdm", "CogniteActivity",  "v1")
TIMESERIES = ViewId("cdf_cdm", "CogniteTimeSeries","v1")
FILE       = ViewId("cdf_cdm", "CogniteFile",      "v1")
WORKORDER  = ViewId(EDM_SPACE, "WorkOrder",              MODEL_VERSION)
EHP        = ViewId(SDM_SPACE, "EquipmentHealthProfile", MODEL_VERSION)


OPS_TABLE = "rwt_Training_TRN_WorkOrderOperations"

print("connected:", client.config.project, "| space:", space)

## §14.2 — Count first

Two cheap calls turn "something feels off" into "two records are missing" —
which is falsifiable, and tells you when you are done.

In [ ]:
WORKORDER = ViewId(EDM_SPACE, "WorkOrder", MODEL_VERSION)

rows   = client.raw.rows.list(db_name=RAW_DB, table_name=OPS_TABLE, limit=-1)
acts   = client.data_modeling.instances.list(sources=ACTIVITY, space=INSTANCE_SPACE, limit=-1)
wo_ids = {n.external_id for n in client.data_modeling.instances.list(
    sources=WORKORDER, space=INSTANCE_SPACE, limit=-1)}
ops    = [a for a in acts if a.external_id not in wo_ids]

print("source rows :", len(rows))    # 8
print("activities  :", len(acts))    # 9  <- WorkOrder implements CogniteActivity
print("operations  :", len(ops))     # 6

## §14.3 — Phantom: a pointer to an empty node

Autocreate materialises a missing target as a bare node. The anti-join therefore
compares against the **view**, not against existence.

In [ ]:
# autoCreateDirectRelations defaults to TRUE, so a bad reference does not dangle --
# CDF CREATES the target as an empty node. "Does it exist?" is the wrong question.
# Ask "is it a real asset?" by comparing against the VIEW.
ASSET = ViewId("cdf_cdm", "CogniteAsset", "v1")

referenced = {}
for n in ops:
    for ref in (n.properties[ACTIVITY].get("assets") or []):
        referenced.setdefault((ref["space"], ref["externalId"]), []).append(n.external_id)

real_assets = {(n.space, n.external_id)
               for n in client.data_modeling.instances.list(
                   sources=ASSET, space=INSTANCE_SPACE, limit=-1)}

phantom = {k: v for k, v in referenced.items() if k not in real_assets}
for (space, xid), holders in phantom.items():
    print(f"  PHANTOM {xid}  <- referenced by {holders}")

print("exists as a node:", len(client.data_modeling.instances.retrieve(
    nodes=(INSTANCE_SPACE, "21-XX-9999")).nodes))
print("assets through the view:", len(real_assets))
# The node EXISTS (1) but is NOT an asset (still 8). Nothing reported an error.

## §14.4 — Orphaned: a node whose parent was never created

Different bug, different owner. This one survived *because* the transformation
uses a LEFT JOIN — an INNER JOIN would have silently dropped it instead.

In [ ]:
orders = {
    n.properties[WORKORDER]["workOrderNumber"]
    for n in client.data_modeling.instances.list(
        sources=WORKORDER, space=INSTANCE_SPACE, limit=-1)
}

for n in ops:
    parent = n.external_id.rsplit("-", 1)[0]
    if parent not in orders:
        print(f"  ORPHAN {n.external_id}  (no work order {parent})")
# expect: WO-9999-0010

## §14.5 — Absent: rows that never became nodes

Invisible from inside the graph — there is nothing to query. The only way to see
them is to compare against the source.

In [ ]:
from collections import Counter

# Map every source row to the external ID it would produce.
derived = {}
for r in rows:
    wo = str(r.columns.get("workOrderNumber") or "").strip()
    op = str(r.columns.get("operationNumber") or "").strip()
    derived[r.key] = f"{wo}-{op.zfill(4)}" if (wo and op) else None

counts = Counter(x for x in derived.values() if x)
actual = {n.external_id for n in ops}

print("rows that produced no external ID at all:")
for k, xid in derived.items():
    if xid is None:
        print(f"   {k}  — blank operationNumber, so concat() returned NULL")

print("rows whose ID collided, so all but one lost deduplication:")
for k, xid in derived.items():
    if xid and counts[xid] > 1:
        print(f"   {k}  ->  {xid}   ({counts[xid]} rows compete for this one ID)")

print("IDs expected but genuinely missing:", sorted(
    {x for x in derived.values() if x} - actual))

# Two findings, two VERY different verdicts:
#   OP-1003-BLANK                     -> blank operationNumber. Source data is wrong.
#   OP-1001-0020 / OP-1001-0020-REV   -> a collision. Deduplication working as designed.
# And "genuinely missing" is EMPTY -- counting IDs would have found nothing.

## §14.7 — Fix at the source, not in the graph

Patch the graph and the next transformation run overwrites you. Fix the row.

In [ ]:
client.raw.rows.insert(
    db_name=RAW_DB, table_name=OPS_TABLE,
    row=RowWrite(key="OP-1001-0030", columns={
        "operationNumber": "0030",
        "workOrderNumber": "WO-1001",
        "tagExternalId":   "21-PA-2001A",     # was 21-XX-9999
        "description":     "Replace outboard bearing",
        "durationHours":   "5",
        "craft":           "MECH",
    }),
)
print("source row corrected — now re-run the transformation:")
print(f"  uv run cdf run transformation tra_{YOURNAME}_Training_TRN_Load_WorkOrderOperations")

Re-run the §14.3 cell. The dangling check should now print nothing — and the orphan should still be there, because you did not touch it.

## §14.8 — The safety net: 72 hours

Delete is not immediate. Instances are soft-deleted, vanish from normal queries,
and are collected about 72 hours later. Inside that window, `/sync` still sees them.

In [ ]:
victim = NodeId(INSTANCE_SPACE, "WO-1002-0010")

def ops_query():
    return QuerySync(
        with_={"ops": NodeResultSetExpression(
            filter=flt.SpaceFilter(INSTANCE_SPACE, "node"), limit=1000)},
        select={"ops": Select([SourceSelector(ACTIVITY, ["name"])])})

# 1. Take a cursor FIRST -- this is the step people skip.
baseline = client.data_modeling.instances.sync(ops_query())
cursor   = baseline.cursors
print("baseline:", len(baseline["ops"]), "instances")

# 2. Now delete.
client.data_modeling.instances.delete(nodes=victim)
print("after delete, retrieve finds:",
      len(client.data_modeling.instances.retrieve(nodes=victim).nodes))

# 3. Sync FROM THAT CURSOR -- the deletion appears, stamped with deletedTime.
sq = ops_query(); sq.cursors = cursor
for n in client.data_modeling.instances.sync(sq)["ops"]:
    print("  changed:", n.external_id, "| deletedTime:", n.deleted_time)

# Contrast: a cursorless sync shows current state and NO deletions at all.
fresh = client.data_modeling.instances.sync(ops_query())
print("fresh sync:", len(fresh["ops"]), "instances,",
      sum(1 for n in fresh["ops"] if n.deleted_time), "carrying deletedTime")

Restore it by re-running the transformation — the source row never moved, so the
upsert recreates the node with the same identity.

**Deleting edges before nodes** matters here: deleting a node cascades to its edges,
and restoring the node does *not* bring them back.

## Gate

- [ ] You can state the 8-vs-6 mismatch and explain all four discrepancies
- [ ] Anti-join found `21-XX-9999` before the fix, and nothing after
- [ ] You found `WO-9999-0010` and can say why it is a different bug
- [ ] You can explain why `OP-1001-0020-REV` being absent is correct
- [ ] You deleted a node, saw it in `/sync` with a `deletedTime`, and restored it

→ **[Chapter 17 — Cross-cutting mastery](../17-cross-cutting-mastery.md)**